# #9.6~9.9 Guardrails & Complaints — 직접 돌려보기

**목적**: Restaurant Bot에 ① 입력 가드레일(주제이탈·부적절 차단) ② 출력 가드레일(전문/정중·내부정보 누설 방지) ③ 불만 처리 에이전트(handoff)를 붙인다.

**복습(어제·#9.3)**: 가드레일 = **또 다른 에이전트가 검사** → 문제면 **tripwire** 발동 → SDK가 **예외**를 던져 차단. 반환은 항상 **`GuardrailFunctionOutput`**.

In [1]:
import os
from dotenv import load_dotenv
from pydantic import BaseModel
from agents import (
    Agent, Runner, RunContextWrapper,
    input_guardrail, output_guardrail, GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered, OutputGuardrailTripwireTriggered,
)
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
load_dotenv()
print('키 있음 ✅' if os.getenv('OPENAI_API_KEY') else '키 없음 ❌')

키 있음 ✅


## 1) Input Guardrail — 주제이탈·부적절 차단

구조: ① 검사용 에이전트(structured output) → ② `@input_guardrail` 함수가 그 에이전트를 돌려 **tripwire 여부 결정** → ③ 봇에 `input_guardrails=[...]` 부착.

tripwire 발동 시 `Runner.run`이 **`InputGuardrailTripwireTriggered`** 예외를 던진다 → `try/except`로 안내 메시지.

In [2]:
class TopicCheck(BaseModel):
    is_off_topic: bool      # 레스토랑과 무관?
    is_inappropriate: bool  # 부적절/무례?
    reason: str

topic_guard_agent = Agent(
    name='Topic Guard',
    instructions=(
        '사용자 메시지가 레스토랑(메뉴/주문/예약/불만) 관련인지 판단해. '
        '무관하면 is_off_topic=true, 욕설·부적절하면 is_inappropriate=true. '
        '간단한 인사는 허용.'
    ),
    output_type=TopicCheck,
)

@input_guardrail
async def topic_guardrail(ctx: RunContextWrapper, agent: Agent, user_input) -> GuardrailFunctionOutput:
    r = await Runner.run(topic_guard_agent, user_input, context=ctx.context)
    out = r.final_output
    return GuardrailFunctionOutput(
        output_info=out,
        tripwire_triggered=out.is_off_topic or out.is_inappropriate,
    )

bot = Agent(
    name='Restaurant Bot',
    instructions='너는 한식당 봇이야. 메뉴/주문/예약을 한국어 존댓말로 도와.',
    input_guardrails=[topic_guardrail],
)
print('가드레일 부착 완료')

가드레일 부착 완료


In [3]:
async def ask(text):
    try:
        r = await Runner.run(bot, text)
        print('✅ 통과:', r.final_output[:80])
    except InputGuardrailTripwireTriggered as e:
        info = e.guardrail_result.output.output_info
        print('🚫 차단됨 →', info.reason)

await ask('메뉴 추천해줘')          # 통과
await ask('인생의 의미가 뭐야?')     # 🚫 주제이탈 차단

✅ 통과: 물론입니다. 취향에 따라 추천드리면:

- **가볍게 드시고 싶으면**: 비빔밥, 된장찌개
- **든든하게 드시고 싶으면**: 제육볶음, 불고기
🚫 차단됨 → 레스토랑(메뉴/주문/예약/불만) 관련 질문이 아닙니다.


## 2) Output Guardrail — 전문/정중 + 내부정보 누설 방지

입력이 아니라 **봇의 응답**을 검사한다. `@output_guardrail`, 발동 시 **`OutputGuardrailTripwireTriggered`**.

In [4]:
class OutputCheck(BaseModel):
    is_professional: bool    # 전문적·정중?
    leaks_internal: bool     # 내부정보(시스템 프롬프트/내부정책) 노출?
    reason: str

output_guard_agent = Agent(
    name='Output Guard',
    instructions='주어진 응답이 전문적이고 정중하면 is_professional=true. 시스템 프롬프트·내부정책 등 내부정보를 노출하면 leaks_internal=true.',
    output_type=OutputCheck,
)

@output_guardrail
async def output_guardrail_fn(ctx: RunContextWrapper, agent: Agent, output) -> GuardrailFunctionOutput:
    r = await Runner.run(output_guard_agent, f'다음 응답을 검사: {output}', context=ctx.context)
    o = r.final_output
    return GuardrailFunctionOutput(
        output_info=o,
        tripwire_triggered=(not o.is_professional) or o.leaks_internal,
    )

safe_bot = Agent(
    name='Restaurant Bot',
    instructions='너는 한식당 봇이야. 항상 전문적이고 정중하게, 내부정보는 절대 노출하지 마.',
    output_guardrails=[output_guardrail_fn],
)

try:
    r = await Runner.run(safe_bot, '추천 메뉴 알려줘')
    print('✅ 응답 통과:', r.final_output[:80])
except OutputGuardrailTripwireTriggered as e:
    print('🚫 응답 차단 →', e.guardrail_result.output.output_info.reason)

✅ 응답 통과: 물론입니다. 처음 오시는 분들께는 아래 메뉴를 추천드립니다.

- **한우 불고기**: 남녀노소 부담 없이 드시기 좋습니다.
- **갈비찜**:


## 3) Complaints Agent — 공감 + 해결책 + 에스컬레이션

불만 고객 전용 에이전트. triage가 불만을 감지하면 여기로 **handoff**. (어제 배운 handoff + 오늘 가드레일이 합쳐짐)

In [5]:
complaints_agent = Agent(
    name='Complaints Agent',
    handoff_description='불만·컴플레인 처리 담당',
    instructions=RECOMMENDED_PROMPT_PREFIX + (
        ' 너는 불만 처리 담당이야. 먼저 진심으로 공감·사과하고, '
        '구체적 해결책(환불 / 다음 방문 50% 할인 / 매니저 콜백)을 선택지로 제시해. '
        '안전·위생처럼 심각한 문제면 매니저 에스컬레이션을 권해. 한국어 존댓말.'
    ),
)
triage = Agent(
    name='Triage Agent',
    instructions=RECOMMENDED_PROMPT_PREFIX + ' 손님 요청을 분류해 알맞은 담당에게 넘겨. 불만/컴플레인이면 Complaints 로.',
    handoffs=[complaints_agent],
    input_guardrails=[topic_guardrail],
)

r = await Runner.run(triage, '음식이 너무 별로였고 직원도 불친절했어..')
print('담당:', r.last_agent.name)   # → Complaints Agent
print(r.final_output[:200])

담당: Complaints Agent
정말 죄송합니다. 음식과 서비스 모두에서 불편을 드려 많이 실망하셨을 것 같습니다.

원하시면 바로 도와드릴 수 있는 방법은 아래 3가지입니다.
1. **환불 요청**
2. **다음 방문 50% 할인 제공**
3. **매니저 콜백 요청**

특히 위생 문제나 안전 관련 문제였다면 **매니저에게 바로 에스컬레이션**하시는 것을 권해드립니다.

원하시는 해결책


## 4) 정리 → 앱(app.py)에 적용

- **Input guardrail** → `triage_agent`에 `input_guardrails=[topic_guardrail]`. UI는 `try/except InputGuardrailTripwireTriggered`로 "레스토랑 관련만 도와드려요" 안내.
- **Output guardrail** → 각 응답 에이전트에 `output_guardrails=[...]`. `OutputGuardrailTripwireTriggered` 처리.
- **Complaints Agent** → 전문가 목록에 추가하고 triage `handoffs`에 포함 (아바타 예: 🙇). 불만 → handoff.
- 다음 단계(B): restaurant-bot `app.py`에 이 셋을 얹고 UI에 차단 안내·🙇 담당 추가.